In [1]:
from transformers import AutoModel, AutoTokenizer, AutoConfig

c = AutoConfig.from_pretrained("distilbert/distilbert-base-multilingual-cased")
c.output_hidden_states = True
m = AutoModel.from_pretrained("distilbert/distilbert-base-multilingual-cased", config=c)
t = AutoTokenizer.from_pretrained("distilbert/distilbert-base-multilingual-cased")

In [27]:
import json
it_text1 = open("italian_ex1.txt", "r").read()

template = "The case concerns the crime of [name of the offence] provided by [eg art x, criminal code] and is referred to the [EIO/EAW/REG 1805]. The countries involved in the cooperation procedure are [x] and [x]. In the case, the relevant ground for refusal(s) is/are [mention the ground for refusal, paying attention to use the same wording of the EU legislation]. The main facts relevant to the case, that triggered the application (or potential application) of the ground for refusal are: [x, y, …]"
prompt = "Summarize the text:<INPUT>{INPUT}</INPUT>\nBe sure to follow this template when creating your summary:<TEMPLATE>{TEMPLATE}</TEMPLATE>\nSummary:"

In [28]:
from summarizer import Summarizer

model_summ = Summarizer(custom_model=m, custom_tokenizer=t, hidden_concat = True, hidden = [-1, -2], gpu_id = 0)
def get_extractive_summary_bert(doc, limit_sentences = 10):
    return model_summ(doc, use_first = False, return_as_list = True, num_sentences = limit_sentences) 

it_ext_sum1 = get_extractive_summary_bert(it_text1, 10)

In [29]:
from env_utils import *
from openai import OpenAI

input_prompt = prompt.format(INPUT=" ".join(it_ext_sum1), TEMPLATE=template)

SYSTEM_PROMPT = f"You are a legal expert. You are tasked with reading legal texts and creating summaries. Your summaries are truthful, relevant, and faithful to the source documents, using only facts and entities present in them, while also including as many as possible. Your summaries read like histories of cases, useful for other lawyers. Your summary must be around 250 words long."
message_history = [
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": input_prompt
                    }
                ]

paras = {
        "model": "meta-llama/Meta-Llama-3-8B-Instruct",
        "temperature": 0,
        "frequency_penalty": 0,
        "presence_penalty": 0,
        "max_tokens": 350
    }

load_env_from_file(".")
client = OpenAI(base_url = "https://api.deepinfra.com/v1/openai", api_key=os.environ["DEEPINFRA_API_KEY"])
chat_completion = client.chat.completions.create(messages=message_history, **paras)


In [30]:
print("Prompt:")
print(input_prompt)
print("Summary:")
print(chat_completion.choices[0].message.content)

Prompt:
Summarize the text:<INPUT>Violazione di legge, in relazione agli artt. della cui mancata osservanza la difesa si è pure doluta. 6, Sentenza n. 931 del 11/01/2018, Yordanov, Rv. D'altro canto, il provvedimento in base al quale è stato emesso il mandato di arresto europeo nei confronti del Tannimi è di certo una decisione giudiziaria esecutiva, che legittima la consegna a mente degli artt. c), della decisione quadro 2002/584/GAI del Consiglio dell'Unione Europea del 13/06/2002 (in questo senso Sez. Infine, è irrilevante la mancata indicazione dell'attività di indagine che l'autorità inglese intende svolgere e in relazione alla quale ha richiesto la consegna del Tamimi. In tale senso si è espressa anche la giurisprudenza di legittimità per la quale, in tema di mandato di arresto europeo, può essere data esecuzione ad una richiesta di consegna nei confronti di persona imputata di un reato per procedere anche solo al suo interrogatorio, atteso che l'art. emesso in relazione ad una c